# Exercise 1: Native and converted script

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Cross-Lingual Transfer · version 0.1.0*

> Compare native/native, converted/converted and native/converted conditions using declared converter directions. Record failed mappings and actual model input scripts.

**Setup.** `pip install sklearn-crfsuite scikit-learn sacrebleu torch transformers sentencepiece fasttext-wheel "stanza==1.11.0" requests "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this. Data come from commit-pinned GitHub repositories and are hash-checked: KazNERD and KyrgyzNER (Exercise 1), SIB-200 (Exercises 2 and 3), the X-WMT Kazakh–Kyrgyz test set (Exercise 4) and the UD 2.17 Turkish–German SAGT and Turkish IMST test files (Exercise 5). Sentence encoders and NLLB-200 are read from local copies (`MODELS_DIR`, `HF_HUB_DIR`, `NLLB_PATH`) or the Hub at the revisions pinned in the code; GlotLID from `GLOTLID_PATH`. Run the notebooks from the `exercises/` folder: the first code cell imports the chapter's helpers from `../code/remaining/transfer.py`. Embeddings, translations and model outputs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`). Everything runs on the CPU.

A CRF tagger (the Sequence Labelling features) is trained on 20,000 KazNERD sentences and tested on the KyrgyzNER test split, both harmonised to PER/LOC/ORG with the chapter's `SCHEMA` (KyrgyzNER's INSTITUTION and BUSINESS count as ORG, declared before scoring). Conversion uses TurkicNLP's Cyrillic→Common Turkic Script (CTS) transliterators, word by word. A second check converts whole sentences and maps the gold spans back with `conversion_map` and `restore_span`.

In [ ]:
import collections, csv, hashlib, io, json, math, os, random, re, statistics, sys
from pathlib import Path
import numpy as np, requests
sys.path.insert(0, str(Path("../code/remaining").resolve()))
from transfer import SCHEMA, bio_spans, conversion_map, restore_span

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_{}.txt"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/data/{}"
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/xwmt/test/"
UD = "https://raw.githubusercontent.com/UniversalDependencies/{}/r2.17/{}"
SHA256 = {
    KAZNERD.format("train"): "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"): "57a35a92139497819a2792d3e9e00a41b387b029b7851be7a64f4968befaddf5",
    UD.format("UD_Turkish_German-SAGT", "qtd_sagt-ud-test.conllu"): "0bbb06de66a490bfbb93d103a03d0d89e3835baf0af755a57b5fa5f885ae01bc",
    UD.format("UD_Turkish-IMST", "tr_imst-ud-test.conllu"): "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
}
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256(path.read_bytes()).hexdigest(); FETCHED[url] = h
    if url in SHA256 and h != SHA256[url]:
        raise ValueError(f"{url} differs from the pinned file")
    return path

_CTS = {}
def to_cts(lang, text):
    """Common Turkic Script view (TurkicNLP transliterator); Turkish and English stay unchanged."""
    if lang in ("tur", "eng"):
        return text
    if lang not in _CTS:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        src = Script.LATIN if lang in ("uzb", "aze") else Script.CYRILLIC
        _CTS[lang] = Transliterator(lang, src, Script.COMMON_TURKIC)
    return _CTS[lang].transliterate(text)

def sib(lang):
    """All 1,004 SIB-200 rows of a language as (index_id, split, topic, text), sorted by index_id."""
    rows = []
    for split in ("train", "dev", "test"):
        text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
        rows += [(int(r["index_id"]), split, r["category"], r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]
    return sorted(rows)

MODELS = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
          "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a"}
NAMES = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "MiniLM",
         "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "XLM-R para", "nllb": "NLLB enc"}
NLLB_PATH = os.environ.get("NLLB_PATH", str(Path("~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M").expanduser()))
NLLB_CODE = {"kaz_Cyrl": "kaz_Cyrl", "kir_Cyrl": "kir_Cyrl", "tat_Cyrl": "tat_Cyrl", "bak_Cyrl": "bak_Cyrl", "uzn_Latn": "uzn_Latn",
             "tur_Latn": "tur_Latn", "rus_Cyrl": "rus_Cyrl", "eng_Latn": "eng_Latn", "fra_Latn": "fra_Latn", "azj_Latn": "azj_Latn"}
def model_source(name):
    rev = MODELS[name]
    for p in (Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser() / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              Path(os.environ.get("MODELS_DIR", CACHE / "models")) / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev
_ENC = {}
def embed(model, texts, lang=None, batch=32):
    """L2-normalised mean-pooled sentence vectors, as in Meaning and Retrieval (same cache)."""
    key = hashlib.sha1(json.dumps([model, lang, texts], ensure_ascii=False).encode()).hexdigest()[:20]
    path = CACHE / "ch13-vectors" / f"{key}.npy"
    if path.exists():
        return np.load(path)
    import torch
    torch.set_num_threads(THREADS)
    if model not in _ENC:
        from transformers import AutoModel, AutoTokenizer
        if model == "nllb":
            from transformers import AutoModelForSeq2SeqLM
            _ENC[model] = (AutoTokenizer.from_pretrained(NLLB_PATH), AutoModelForSeq2SeqLM.from_pretrained(NLLB_PATH).get_encoder().eval(), 512)
        else:
            src, rev = model_source(model)
            _ENC[model] = (AutoTokenizer.from_pretrained(src, revision=rev), AutoModel.from_pretrained(src, revision=rev).eval(), 128)
    tok, enc, limit = _ENC[model]
    if model == "nllb":
        tok.src_lang = NLLB_CODE[lang]
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], padding=True, truncation=True, max_length=limit, return_tensors="pt")
            h = enc(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            out.append(torch.nn.functional.normalize((h * m).sum(1) / m.sum(1).clamp(min=1), dim=-1).numpy())
    vec = np.concatenate(out).astype(np.float32)
    path.parent.mkdir(parents=True, exist_ok=True); np.save(path, vec)
    return vec

def macro_f1(gold, pred, labels):
    from sklearn.metrics import f1_score
    return f1_score(gold, pred, labels=labels, average="macro", zero_division=0)

print("helpers loaded from code/remaining/transfer.py; schema map:", SCHEMA)

In [ ]:
import sklearn_crfsuite
def read_conll(path):
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith("# "):
            c = line.split("\t") if "\t" in line else line.split()
            words.append(c[0]); tags.append(c[-1].strip())
    return sents
EXTRA = {"INSTITUTION": "ORG", "BUSINESS": "ORG"}      # KyrgyzNER types counted as ORG, declared before scoring (as in Sequence Labelling)
def harmonise(tags):
    out = []
    for t in tags:
        m = (SCHEMA.get(t[2:]) or EXTRA.get(t[2:])) if t != "O" else None
        out.append(f"{t[0]}-{m}" if m else "O")
    return out
def spans(tags):
    out, start, typ = set(), None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if t.startswith("I-") and typ == t[2:]:
            continue
        if typ is not None: out.add((start, i, typ))
        start, typ = (i, t[2:]) if t.startswith("B-") else (None, None)
    return out
def prf(gold, pred):
    g = sum(map(len, gold)); p = sum(map(len, pred)); c = sum(len(a & b) for a, b in zip(gold, pred))
    P = c / p if p else 0; R = c / g if g else 0
    return round(100 * P, 1), round(100 * R, 1), round(100 * (2 * P * R / (P + R) if P + R else 0), 1)
def features(words, i):
    w = words[i]; f = {"bias": 1.0, "w": w.lower(), "title": w[:1].isupper(), "upper": w.isupper(),
                       "digit": any(c.isdigit() for c in w), "len": min(len(w), 12)}
    for k in (1, 2, 3, 4):
        f[f"suf{k}"] = w[-k:].lower(); f[f"pre{k}"] = w[:k].lower()
    for d in (-2, -1, 1, 2):
        j = i + d
        f[f"w{d}"] = words[j].lower() if 0 <= j < len(words) else "<pad>"
        f[f"t{d}"] = words[j][:1].isupper() if 0 <= j < len(words) else False
    return f
_memo = {}
def convert(lang, words):                     # word by word, so tokens and gold tags stay aligned
    return [_memo.setdefault((lang, w), to_cts(lang, w)) for w in words]
kaz = read_conll(fetch(KAZNERD.format("train")))
random.Random(0).shuffle(kaz); kaz = kaz[:20000]
kir = read_conll(fetch(KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt")))
gold = [spans(harmonise(t)) for _, t in kir]
print(f"source: 20,000 random KazNERD training sentences; target: all {len(kir):,} KyrgyzNER test sentences, "
      f"{sum(map(len, gold)):,} PER/LOC/ORG mentions")
print("Kyrgyz CTS input, first words:", " ".join(convert("kir", kir[0][0][:8])))
crfs = {}
for cond in ("native", "CTS"):
    X = [[features(convert("kaz", ws) if cond == "CTS" else ws, i) for i in range(len(ws))] for ws, _ in kaz]
    crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100)
    crf.fit(X, [harmonise(t) for _, t in kaz]); crfs[cond] = crf
print(f"{'train script':<14}{'test script':<13}{'P':>6}{'R':>6}{'F1':>6}   F1 PER / LOC / ORG")
for train, test in (("native", "native"), ("CTS", "CTS"), ("native", "CTS"), ("CTS", "native")):
    Xt = [[features(convert("kir", ws) if test == "CTS" else ws, i) for i in range(len(ws))] for ws, _ in kir]
    pred = [spans(p) for p in crfs[train].predict(Xt)]
    P, R, F = prf(gold, pred)
    by = [prf([{x for x in s if x[2] == t} for s in gold], [{x for x in s if x[2] == t} for s in pred])[2] for t in ("PER", "LOC", "ORG")]
    print(f"{train:<14}{test:<13}{P:6.1f}{R:6.1f}{F:6.1f}   " + " / ".join(f"{v:.1f}" for v in by))
# Converting whole sentences instead: which gold character spans can be mapped back with the chapter's helpers?
ok = bad = invalid = 0
for ws, tags in kir:
    text = " ".join(ws); pos = 0; words = []
    for w, t in zip(ws, harmonise(tags)):
        words.append({"start": pos, "end": pos + len(w), "text": w, "tag": t}); pos += len(w) + 1
    try:
        gold_spans = bio_spans(words, text)
    except ValueError:
        invalid += 1; continue
    if not gold_spans: continue
    conv, back = conversion_map(text, lambda s: to_cts("kir", s))
    fwd = {v: k for k, v in back.items()}
    for s, e, typ in gold_spans:
        try:
            ok += restore_span((fwd[s], fwd[e], typ), back) == (s, e, typ)
        except (KeyError, ValueError):
            bad += 1
print(f"sentence-level conversion: {ok:,} gold spans map back to their exact offsets, {bad} do not; "
      f"{invalid} sentences rejected by bio_spans (I- tag without a matching B-)")

## Worked solution

Output of `experiments/ch16_exercises/run_ch16_exercises.py`:

```
source: 20,000 random KazNERD training sentences; target: all 3,867 KyrgyzNER test sentences, 3,091 PER/LOC/ORG mentions
Kyrgyz CTS input, first words: Tacikstanda Mamlekettik veterinardık közömöl kızmatı ölkögö adal emes
train script  test script       P     R    F1   F1 PER / LOC / ORG
native        native         44.8  24.8  31.9   41.5 / 34.9 / 6.2
CTS           CTS            48.5  26.9  34.6   42.6 / 38.4 / 13.0
native        CTS             6.9   6.9   6.9   21.3 / 6.3 / 1.3
CTS           native         35.9  17.2  23.2   37.7 / 12.3 / 0.0
sentence-level conversion: 3,090 gold spans map back to their exact offsets, 1 do not; 1 sentences rejected by bio_spans (I- tag without a matching B-)
```

**Conversion changes this CRF's scores.** Native/native span F1 is 31.9, CTS/CTS is 34.6, native/CTS is 6.9 and CTS/native is 23.2. These results concern one source sample and feature set. The experiment does not isolate which lexical or contextual features explain the differences, or how much domain and annotation conventions contribute.

The NER score uses token-index spans after word-by-word conversion. Its local parser ignores orphan `I` tags. The separate character-mapping diagnostic uses the stricter `bio_spans` helper, which rejects one sentence; 3,090 gold spans map back and one fails. It constructs text by joining words with spaces, so those offsets do not verify preservation of the original document spacing. Nor does mapping gold spans test restoration of model-predicted spans.

The 2.7-point native/native versus CTS/CTS difference has no paired uncertainty estimate or repeated source sampling. Test those before drawing a broader conclusion.

## Your turn — extension

Repeat the comparison with Kazakh→Tatar or Kazakh→Bashkir (where the Cyrillic alphabets differ more), use three random samples of the source data, and report paired differences with a bootstrap interval over target sentences.